In [1]:
from datasets import load_from_disk

/home/miguel/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ds = load_from_disk("dataset/canada-rec-split-updated")
print(ds)

DatasetDict({
    train: Dataset({
        features: ['PK', 'Text', 'Speakers', 'Interventions', 'label'],
        num_rows: 1229
    })
    dev: Dataset({
        features: ['PK', 'Text', 'Speakers', 'Interventions', 'label'],
        num_rows: 178
    })
    test: Dataset({
        features: ['PK', 'Text', 'Speakers', 'Interventions', 'label'],
        num_rows: 186
    })
})


In [4]:
ds["test"][10]["Text"]

'INITIATIVE REGARDING FIGHTING AGAINST FORCED LABOUR AND CHILD LABOUR IN SUPPLY CHAINS ACT, SPONSORED BY HON. JOHN MCKAY, MR. ARNOLD VIERSEN, MR. SIMON-PIERRE SAVARD-TREMBLAY, MR. PETER JULIAN, MR. TERRY SHEEHAN, AND MR. TED FALK.\n\n**The Speaker:**\nWe shall now proceed to the next item on the order of business, regarding the Fighting Against Forced Labour and Child Labour in Supply Chains Act, also known as Bill S-211. This bill aims to address the pressing issue of forced and child labour within global supply chains by mandating federal departments and large companies to report on their efforts to prevent such practices.\n\n**Context prior to the debate (Highlight):**\nThe debate surrounding Bill S-211 has revealed a significant divide among parliamentarians. Proponents argue that the bill is a crucial first step towards transparency and accountability in supply chains, while opponents contend that it lacks the necessary enforcement mechanisms and fails to impose binding obligation

In [10]:
from datasets import DatasetDict
ds_update = ds.copy()
# to dataset dict
ds_update = DatasetDict({
    "train": ds_update["train"],
    "dev": ds_update["dev"],
    "test": ds_update["test"]
})
print(ds_update)

DatasetDict({
    train: Dataset({
        features: ['PK', 'Text', 'Speakers', 'Interventions', 'label'],
        num_rows: 1229
    })
    dev: Dataset({
        features: ['PK', 'Text', 'Speakers', 'Interventions', 'label'],
        num_rows: 178
    })
    test: Dataset({
        features: ['PK', 'Text', 'Speakers', 'Interventions', 'label'],
        num_rows: 186
    })
})


In [11]:
import os
from openai import OpenAI

import os
from dotenv import load_dotenv
load_dotenv()

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))


def generar_resumen_parlamentario(texto_debate) -> str:
    instrucciones_sistema = (
        "You are the Speaker of the House in a Parliament. Your goal is to read the transcript "
        "of a debate and generate a highly formal 'initiative header' that serves as an "
        "explanatory introduction before reading the full debate.\n\n"
        "Your response MUST have the following strict structure and tone:\n\n"
        "INITIATIVE REGARDING [MAIN TOPIC IN CAPITAL LETTERS], "
        "SPONSORED BY [NAMES OF THE POLITICIANS OR PARTIES INVOLVED IN CAPITAL LETTERS].\n\n"
        "**The Speaker:**\n"
        "We shall now proceed to the next item on the order of business, regarding [formal and neutral "
        "explanation of 2 to 3 lines about the central topic to be discussed, mentioning the core of the debate].\n\n"
        "**Context prior to the debate (Highlight):**\n"
        "[A brief paragraph objectively summarizing the conflict: what one side accuses and what the other defends]."
    )

    try:
        # Llamada a la API usando el modelo gpt-4o-mini (rápido y económico para textos)
        answer = client.chat.completions.create(
            model="gpt-4o-mini", 
            messages=[
                {"role": "system", "content": instrucciones_sistema},
                {"role": "user", "content": f"Please summarize the following session:\n\n{texto_debate}"}
            ],
            temperature=0,
            seed=123,
        )

        return answer.choices[0].message.content
        
    except Exception as e:
        print(f"Error al procesar la solicitud: {e}")

In [12]:
ds_update

DatasetDict({
    train: Dataset({
        features: ['PK', 'Text', 'Speakers', 'Interventions', 'label'],
        num_rows: 1229
    })
    dev: Dataset({
        features: ['PK', 'Text', 'Speakers', 'Interventions', 'label'],
        num_rows: 178
    })
    test: Dataset({
        features: ['PK', 'Text', 'Speakers', 'Interventions', 'label'],
        num_rows: 186
    })
})

In [15]:
import tqdm

def añadir_cabecera(ejemplo):
    texto_debate = ejemplo["Text"]
    
    # Llamamos a tu función de la API
    resumen_generado = generar_resumen_parlamentario(texto_debate)
    
    # Sobrescribimos la columna "Text" con el nuevo formato
    ejemplo["Text"] = resumen_generado + "\n" + texto_debate
    
    return ejemplo

# 2. Aplicamos la función a los splits
for split in ["dev", "test"]:
    print(f"\n--- Generando resúmenes para el split: {split} ---")
    ds_update[split] = ds_update[split].map(añadir_cabecera)
    
    #dataset_prueba = ds_update[split].select([0]) # Solo coge el índice 0
    #dataset_prueba = dataset_prueba.map(añadir_cabecera)
    
    # Imprimimos para verificar que se guardó bien
    # print(f"Resultado final en el dataset para {split} sample 0:\n")
    # print(dataset_prueba[0]["Text"])


--- Generando resúmenes para el split: dev ---


Map: 100%|██████████| 178/178 [25:23<00:00,  8.56s/ examples]



--- Generando resúmenes para el split: test ---


Map: 100%|██████████| 186/186 [26:30<00:00,  8.55s/ examples]


In [16]:
# save updated dataset
ds_update.save_to_disk("dataset/canada-rec-split-updated")

Saving the dataset (1/1 shards): 100%|██████████| 186/186 [00:00<00:00, 13272.44 examples/s]


## Estadisticas del dataset

In [16]:
from datasets import load_from_disk
ds = load_from_disk("dataset/canada-rec-split")
print(ds)

DatasetDict({
    train: Dataset({
        features: ['PK', 'Text', 'Speakers', 'Interventions', 'label'],
        num_rows: 1229
    })
    dev: Dataset({
        features: ['PK', 'Text', 'Speakers', 'Interventions', 'label'],
        num_rows: 178
    })
    test: Dataset({
        features: ['PK', 'Text', 'Speakers', 'Interventions', 'label'],
        num_rows: 186
    })
    tiny: Dataset({
        features: ['PK', 'Text', 'Speakers', 'Interventions', 'label'],
        num_rows: 50
    })
})


In [ ]:
def get_stats(ds):
    
    # contar el numero media de muestras positivas 
    num_samples = len(ds)
    num_positive_samples = sum([sum(sample["label"]) for sample in ds])
    avg_positive_samples = num_positive_samples / num_samples
    print(f"Numero total de muestras: {num_samples}")
    print(f"Numero total de muestras positivas: {num_positive_samples}")
    print(f"Numero medio de muestras positivas: {avg_positive_samples:.2f}")

import nltk
nltk.download('punkt')
from nltk.tokenize import word_tokenize

def count_words(text, lang='english'):
    tokens = word_tokenize(text, language=lang)
    return len(tokens)
    #return len(text.split())

In [19]:
from datasets import concatenate_datasets
ds_full = concatenate_datasets([ds["train"], ds["dev"], ds["test"]])
print(ds_full)

# contar el numero total de palbras en el dataset completo
total_words = sum([count_words(sample["Text"]) for sample in ds_full])
print(f"Numero total de palabras en el dataset completo: {total_words}")

Dataset({
    features: ['PK', 'Text', 'Speakers', 'Interventions', 'label'],
    num_rows: 1593
})
Numero total de palabras en el dataset completo: 13346492


In [20]:
# sacar la media de palabras por intervencion "Interventions"
total_words = 0
total_interventions = 0

for sample in ds_full:
    interventions = sample["Interventions"]
    for intervention in interventions:
        total_words += sum([count_words(text) for text in intervention])
        total_interventions += 1

avg_words_per_intervention = total_words / total_interventions if total_interventions > 0 else 0
print(f"Media de palabras por intervención: {avg_words_per_intervention:.2f}")

Media de palabras por intervención: 548.23


In [29]:
ds = load_from_disk("dataset/canada-rec")
print(ds)
#print(sum(ds[0]["label"]))

get_stats(ds)

Dataset({
    features: ['PK', 'Text', 'Speakers', 'Interventions', 'label'],
    num_rows: 1593
})
Numero total de muestras: 1593
Numero total de muestras positivas: 23449
Numero medio de muestras positivas: 14.72


In [13]:
print(ds[0]['PK'])

6596_4


In [10]:
ds = load_from_disk("dataset/canada-rec")
print(ds)

#contar interventions
count = 0
for i in ds:
    count += len(i['Interventions'])
print(count)

Dataset({
    features: ['PK', 'Text', 'Speakers', 'Interventions', 'label'],
    num_rows: 1593
})
23449


In [34]:
print(ds["test"][0]["Text"])

Esta sesión del parlamento se realizó el 2024-05-07. 11L/PO/P-0750 PREGUNTA DEL SEÑOR DIPUTADO DON NICASIO JESÚS GALVÁN SASIA, DEL GRUPO PARLAMENTARIO VOX, SOBRE MEDIDAS QUE SE VAN A LLEVAR A CABO PARA DEMOCRATIZAR Y REDISTRIBUIR LA RIQUEZA DEL SECTOR TURÍSTICO, DIRIGIDA A LA PRESIDENCIA DEL GOBIERNO La señora PRESIDENTA: Siguiente pregunta, del señor diputado don Nicasio Galván Sasia, del Grupo Parlamentario VOX, sobre medidas que se van a llevar a cabo para democratizar y redistribuir la riqueza del sector turístico, dirigida al señor presidente del Gobierno. Cuando quiera. El señor GALVÁN SASIA (desde su escaño): Buenos días, señor Clavijo, buenos días. Escuchándole en la rueda de prensa posterior a la Conferencia de Presidentes nos han surgido varias preguntas, y nos consta que no solo a nosotros. Se le oía escuchar hablar de la democratización y la redistribución de la riqueza del sector turístico y culpabilizó al turismo -sí, a esas 500 000 personas que han venido a vivir a Canar

In [52]:
id = 590
print(ds["test"][id]["Speakers"])
print(ds["test"][id]["Text"])

['Navarro De Paz', 'Rivero Baute']
Esta sesión del parlamento se realizó el 2013-03-26. · 8L/PO/P-0826 Pregunta urgente, de la señora diputada doña María Australia Navarro de Paz, del Grupo Parlamentario Popular, sobre recursos humanos y económicos destinados para oponerse judicialmente a los permisos de investigación de hidrocarburos, dirigida al señor presidente del Gobierno (Continuación). El señor presidente: Señorías, tal como establece el Reglamento, vamos a darles cuenta de un escrito del Grupo Parlamentario Popular, al amparo de los artículos 82.2 y 82.3 y siguientes del Reglamento del Parlamento de Canarias. Y tal como establece el propio Reglamento tiene la palabra, por tiempo de tres minutos, su portavoz, doña María Australia Navarro de Paz. La señora Navarro de Paz (Desde su escaño): Gracias, presidente. Señor presidente del Gobierno: una vez más perdió los papeles en este hemiciclo; una vez más olvidó el papel institucional que se le presume a un presidente, al presidente 

In [50]:
inter = ds["test"][id]["Interventions"]
for i, intervention in enumerate(inter):
    print(f"Intervention {i}:")
    for speaker in intervention:
        print(speaker)
        print("***")
    print("-----")


Intervention 0:
Gracias, presidente. Señor presidente del Gobierno: una vez más perdió los papeles en este hemiciclo; una vez más olvidó el papel institucional que se le presume a un presidente, al presidente de todos los canarios; y una vez más, tras injuriar, huyó a toda prisa de este salón para no escuchar la respuesta que merecen sus mentiras. Pero, mire, señor Rivero, hoy contesto, hoy le contesto, y espero que no se vuelva a escapar corriendo. Señor presidente, acusó a mi grupo, a mi grupo político, de beneficiarse con nuestra defensa clara de los beneficios del petróleo y del gas para Canarias y para los canarios, porque no quería confesar los miles y miles de euros de los canarios que lleva gastados en pleitos inútiles que siempre fracasan. Mire usted, señor Rivero, respete el cargo que ostenta. Sí, señor Rivero. Su comportamiento en el último Pleno fue mezquino y cobarde. Sí, fue mezquino porque, a falta de argumentos convincentes, recurrió a la insidia, a la mentira y a la de

In [33]:
vector = ds["test"][0]["label"]
# cuenta la cantidad de elementos no nulos
non_zero_count = sum(1 for v in vector if v != 0)
print(f"Cantidad de elementos no nulos en el vector de etiquetas: {non_zero_count}")

Cantidad de elementos no nulos en el vector de etiquetas: 2


In [31]:
import numpy as np
# --- CÁLCULO AUTOMÁTICO DEL PRIOR (Insertar antes del Trainer) ---
print("\n[-] Calculando el Prior REAL de la estrategia actual...")
total_ones = 0
total_elements = 0
train_dataset = ds['train']

# Iteramos sobre una muestra del dataset procesado (o todo si es rápido)
# train_dataset ya tiene los vectores "inflados" con all_participants
for i in range(min(1000, len(train_dataset))): # Muestreamos 1000 ejemplos para ir rápido
    labels = np.array(train_dataset[i]['label']) # Esto es un tensor
    total_ones += labels.sum().item()
    total_elements += len(labels)

real_prior = total_ones / total_elements
print(f"    Prior Observado: {real_prior:.5f} (es decir, {real_prior*100:.2f}%)")
print(f"    Recomendación para nnPU: Usar pi = {real_prior * 1.1:.5f} (un poco de margen)")

# Sobreescribimos el valor para usarlo en el Trainer
prior_to_use = real_prior * 1.1
print(f"    Usando pi = {prior_to_use:.5f} en el Trainer.\n")


[-] Calculando el Prior REAL de la estrategia actual...
    Prior Observado: 0.01155 (es decir, 1.16%)
    Recomendación para nnPU: Usar pi = 0.01271 (un poco de margen)
    Usando pi = 0.01271 en el Trainer.



In [66]:
# concate todo el dataset 
from datasets import concatenate_datasets
ds = load_from_disk("dataset/canada-rec-split")
full_dataset = concatenate_datasets([ds['train'], ds['dev'], ds['test']])

# cuenta la media de 1 que hay en el vector de etiquetas
total_ones = 0
for i in range(len(full_dataset)):
    labels = np.array(full_dataset[i]['label']) # Esto es un tensor
    total_ones += labels.sum().item()

avg = total_ones / len(full_dataset)

print(f"Media de MP que hablan en cada iniciativa: {np.round(avg, 4)}. su desviación estándar es: {np.round(np.std([np.array(full_dataset[i]['label']).sum().item() for i in range(len(full_dataset))]), 4)}")
print(f"Maximo de MP que hablan en una iniciativa: {max([np.array(full_dataset[i]['label']).sum().item() for i in range(len(full_dataset))])}")

Media de MP que hablan en cada iniciativa: 14.72. su desviación estándar es: 12.95
Maximo de MP que hablan en una iniciativa: 58


In [1]:
import torch
from sentence_transformers import SentenceTransformer

# Nombre del modelo (asegúrate de que el modelo base soporte la arquitectura Qwen2/3)
model_name = "Qwen/Qwen3-Embedding-8B" 

# Configuración de cuantización para BitsAndBytes
# Esto carga el modelo en 4 bits directamente en la VRAM
quantization_config = {
    "load_in_4bit": True,
    "bnb_4bit_compute_dtype": torch.float16, # Usar float16 para cálculos
    "bnb_4bit_quant_type": "nf4",            # Normal Float 4 (mejor rendimiento que fp4)
    "bnb_4bit_use_double_quant": True        # Ahorra un poco más de memoria
}

print("📥 Cargando modelo en 4-bit...")

# Cargamos el modelo pasando los argumentos de cuantización
model = SentenceTransformer(
    model_name,
    device="cuda", # Necesario para bitsandbytes
    model_kwargs=quantization_config
)

print("✅ Modelo cargado exitosamente en 4-bit.")

/home/miguel/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


📥 Cargando modelo en 4-bit...


The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.
Loading checkpoint shards: 100%|██████████| 4/4 [00:27<00:00,  6.79s/it]


✅ Modelo cargado exitosamente en 4-bit.


In [2]:
from sentence_transformers.util import cos_sim

# Definimos oraciones de prueba
sentences = [
    "El aprendizaje automático es fascinante.",
    "La inteligencia artificial está transformando el mundo.",
    "Me gusta comer pizza los viernes."
]

# Codificamos (esto usará el modelo cuantizado en GPU)
embeddings = model.encode(sentences)

print(f"\nDimensiones de los embeddings: {embeddings.shape}")

# Calculamos similitud para verificar coherencia
sim_1_2 = cos_sim(embeddings[0], embeddings[1])
sim_1_3 = cos_sim(embeddings[0], embeddings[2])

print(f"Similitud (IA vs ML): {sim_1_2.item():.4f}")  # Debería ser alta
print(f"Similitud (IA vs Pizza): {sim_1_3.item():.4f}") # Debería ser baja


Dimensiones de los embeddings: (3, 4096)
Similitud (IA vs ML): 0.6836
Similitud (IA vs Pizza): 0.3870
